# import data

In [19]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter      # ganti sesuai provider                     # atau langchain_qdrant, langchain_community.vectorstores.FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


In [20]:
import pandas as pd
data = pd.read_json('/Users/muhammadzuamaalamin/Documents/RisetTextMining/retrieval2/datacorpus.json')
data

,id,pasal,bab,judul,context,ayat,buku,bagian,paragraf
0,kuhp_pasal_1,1,BAB I - RUANG LINGKUP BERLAKUNYA KETENTUAN PER...,,Pasal 1\n(1) Tidak ada satu perbuatan pun yang...,"[{'nomor': '1', 'isi': '(1) Tidak ada satu per...",NaN,NaN,NaN
1,kuhp_pasal_2,2,BAB I - RUANG LINGKUP BERLAKUNYA KETENTUAN PER...,,Pasal 2\n(1) Ketentuan sebagaimana dimaksud da...,"[{'nomor': '1', 'isi': '(1) Ketentuan sebagaim...",NaN,NaN,NaN
2,kuhp_pasal_3,3,BAB I - RUANG LINGKUP BERLAKUNYA KETENTUAN PER...,,Pasal 3\n(1) Dalam hal terdapat perubahan pera...,"[{'nomor': '1', 'isi': '(1) Dalam hal terdapat...",NaN,NaN,NaN
3,kuhp_pasal_4,4,BAB I - RUANG LINGKUP BERLAKUNYA KETENTUAN PER...,,Pasal 4\nKetentuan pidana dalam Undang-Undang ...,[],NaN,NaN,NaN
4,kuhp_pasal_5,5,BAB I - RUANG LINGKUP BERLAKUNYA KETENTUAN PER...,,Pasal 5\nKetentuan pidana dalam Undang-Undang ...,[],NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
616,kuhp_pasal_617,617,BAB XXXVI - KETENTUAN PERALIHAN,,Pasal 617\nPada saat Undang-Undang ini mulai b...,[],BUKU KEDUA - TINDAK PIDANA,,NaN
617,kuhp_pasal_618,618,BAB XXXVI - KETENTUAN PERALIHAN,,Pasal 618\nPada saat Undang-Undang ini mulai b...,[],BUKU KEDUA - TINDAK PIDANA,,NaN
618,kuhp_pasal_619,619,BAB XXXVI - KETENTUAN PERALIHAN,,Pasal 619\nPada saat Undang-Undang ini mulai b...,[],BUKU KEDUA - TINDAK PIDANA,,NaN
619,kuhp_pasal_620,620,BAB XXXVI - KETENTUAN PERALIHAN,,Pasal 620\nPada saat Undang-Undang ini mulai b...,[],BUKU KEDUA - TINDAK PIDANA,,NaN


In [21]:
data = data.drop(columns=['id','pasal','bab','judul','ayat','buku','bagian','paragraf'])
data

,context
0,Pasal 1\n(1) Tidak ada satu perbuatan pun yang...
1,Pasal 2\n(1) Ketentuan sebagaimana dimaksud da...
2,Pasal 3\n(1) Dalam hal terdapat perubahan pera...
3,Pasal 4\nKetentuan pidana dalam Undang-Undang ...
4,Pasal 5\nKetentuan pidana dalam Undang-Undang ...
...,...
616,Pasal 617\nPada saat Undang-Undang ini mulai b...
617,Pasal 618\nPada saat Undang-Undang ini mulai b...
618,Pasal 619\nPada saat Undang-Undang ini mulai b...
619,Pasal 620\nPada saat Undang-Undang ini mulai b...


In [22]:
data['context'] = (data['context']
        .str.replace("\n", " ")
        .str.replace(r" +", " ", regex=True)
        .str.strip())
data

,context
0,Pasal 1 (1) Tidak ada satu perbuatan pun yang ...
1,Pasal 2 (1) Ketentuan sebagaimana dimaksud dal...
2,Pasal 3 (1) Dalam hal terdapat perubahan perat...
3,Pasal 4 Ketentuan pidana dalam Undang-Undang b...
4,Pasal 5 Ketentuan pidana dalam Undang-Undang b...
...,...
616,Pasal 617 Pada saat Undang-Undang ini mulai be...
617,Pasal 618 Pada saat Undang-Undang ini mulai be...
618,Pasal 619 Pada saat Undang-Undang ini mulai be...
619,Pasal 620 Pada saat Undang-Undang ini mulai be...


In [24]:
# Chunking — hasilkan list of Document, bukan plain string
splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,
    chunk_overlap=50
)

documents = []

for pasal in data['context']:
    if len(pasal) > 4000:
        # split_text → list[str], bungkus jadi Document
        splits = splitter.split_text(pasal)
        documents.extend([Document(page_content=s) for s in splits])
    else:
        documents.append(Document(page_content=pasal))


In [25]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import os
# Load embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cpu"}
)

db_path = "faiss_indexallminiLm"

# %%
# Buat atau load FAISS index
if not os.path.exists(db_path):
    vectorstore = FAISS.from_documents(documents, embeddings)  # ✅ Document objects
    vectorstore.save_local(db_path)
else:
    vectorstore = FAISS.load_local(
        db_path,
        embeddings,
        allow_dangerous_deserialization=True
    )


In [26]:
for i in range(len(documents)):
    print(documents[i].page_content)

Pasal 1 (1) Tidak ada satu perbuatan pun yang dapat dikenai sanksi pidana dan/atau tindakan, kecuali atas kekuatan peraturan pidana dalam peraturan perundang-undangan yang telah ada sebelum perbuatan dilakukan. (2) Dalam menetapkan adanya Tindak Pidana dilarang digunakan analogi.
Pasal 2 (1) Ketentuan sebagaimana dimaksud dalam Pasal 1 ayat (1) tidak mengurangi berlakunya hukum yang hidup dalam masyarakat yang menentukan bahwa seseorang patut dipidana walaupun perbuatan tersebut tidak diatur dalam Undang-Undang ini. (2) Hukum yang hidup dalam masyarakat sebagaimana dimaksud pada ayat (1) berlaku dalam tempat hukum itu hidup dan sepanjang tidak diatur dalam Undang- Undang ini dan sesuai dengan nilai-nilai yang terkandung dalam Pancasila, Undang-Undang Dasar Negara Republik Indonesia Tahun 1945, hak asasi manusia, dan asas hukum umum yang diakui masyarakat bangsa-bangsa. (3) Ketentuan mengenai tata cara dan kriteria penetapan hukum yang hidup dalam masyarakat diatur dengan Peraturan Peme

In [27]:
query = "Kalau menghina Presiden di media sosial, ancamannya berapa?"

results = vectorstore.similarity_search_with_score(query, k=10)

for i, (doc, score) in enumerate(results, 1):
    print(f"\nResult {i} | score={score:.4f}")
    print(doc.page_content[:400])



Result 1 | score=18.4832
Pasal 218 (1) Setiap Orang yang Di Muka Umum menyerang kehormatan atau harkat dan martabat diri Presiden dan/atau Wakil Presiden, dipidana dengan pidana penjara paling lama 3 (tiga) tahun atau pidana denda paling banyak kategori IV. (2) Tidak merupakan penyerangan kehormatan atau harkat dan martabat sebagaimana dimaksud pada ayat (1), jika perbuatan dilakukan untuk kepentingan umum atau pembelaan 

Result 2 | score=18.8488
Pasal 217 Setiap Orang yang menyerang diri Presiden dan/atau Wakil Presiden yang tidak termasuk dalam ketentuan pidana yang lebih berat, dipidana dengan pidana penjara paling lama 5 (lima) tahun.

Result 3 | score=19.3270
Pasal 219 Setiap Orang yang menyiarkan, mempertunjukkan, atau menempelkan tulisan atau gambar sehingga terlihat oleh umum, memperdengarkan rekaman sehingga terdengar oleh umum, atau menyebarluaskan dengan sarana teknologi informasi yang berisi penyerangan kehormatan atau harkat dan martabat terhadap Presiden dan/atau Wakil 

In [28]:
from langchain.retrievers import BM25Retriever, EnsembleRetriever

bm25 = BM25Retriever.from_documents(documents)
bm25.k = 5

vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25, vector_retriever],
    weights=[0.5, 0.5]
)

In [29]:
query = "Kalau menghina Presiden di media sosial, ancamannya berapa?"

docs = hybrid_retriever.get_relevant_documents(query)

for i, (doc, score) in enumerate(results, 1):
    print(f"\nResult {i} | score={score:.4f}")
    print(f"Halaman: {doc.metadata.get('page')}")
    print(doc.page_content[:400])



Result 1 | score=18.4832
Halaman: None
Pasal 218 (1) Setiap Orang yang Di Muka Umum menyerang kehormatan atau harkat dan martabat diri Presiden dan/atau Wakil Presiden, dipidana dengan pidana penjara paling lama 3 (tiga) tahun atau pidana denda paling banyak kategori IV. (2) Tidak merupakan penyerangan kehormatan atau harkat dan martabat sebagaimana dimaksud pada ayat (1), jika perbuatan dilakukan untuk kepentingan umum atau pembelaan 

Result 2 | score=18.8488
Halaman: None
Pasal 217 Setiap Orang yang menyerang diri Presiden dan/atau Wakil Presiden yang tidak termasuk dalam ketentuan pidana yang lebih berat, dipidana dengan pidana penjara paling lama 5 (lima) tahun.

Result 3 | score=19.3270
Halaman: None
Pasal 219 Setiap Orang yang menyiarkan, mempertunjukkan, atau menempelkan tulisan atau gambar sehingga terlihat oleh umum, memperdengarkan rekaman sehingga terdengar oleh umum, atau menyebarluaskan dengan sarana teknologi informasi yang berisi penyerangan kehormatan atau harkat dan 

# model

In [30]:
from langchain.prompts import ChatPromptTemplate

# ── Versi Ringan untuk Model 4B ───────────────────────────────────────────
prompt_template = ChatPromptTemplate.from_template("""
Kamu adalah ahli hukum KUHP Indonesia.
Jawab berdasarkan konteks berikut saja.

Konteks:
{context}

Pertanyaan:
{question}

Jawaban (sebutkan nomor pasal dan ancaman hukumannya):
""")

# model 1b

In [31]:
from langchain.chains import RetrievalQA
from langchain_ollama import ChatOllama

llm = ChatOllama(model="gemma3:4b", temperature=0.1)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=hybrid_retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt_template},
    return_source_documents=True
)
response = qa_chain.invoke(
    {"query": "Berapa denda maksimal kalau ketahuan main judi online??"}
)

print(response["result"])


Berdasarkan konteks yang diberikan, denda maksimal jika ketahuan main judi online adalah **Rp50.000.000,00 (lima puluh juta rupiah)**. Ini tercantum dalam Pasal 427 ayat (2) yang menyatakan "pidana denda paling banyak kategori III".



In [32]:
from langchain.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template("""
Kamu adalah ahli hukum KUHP Indonesia.
Jawab HANYA berdasarkan konteks dokumen di bawah ini.

Konteks:
{context}

Pertanyaan:
{question}

Instruksi jawaban:
- Sebutkan semua pasal yang relevan, bisa lebih dari satu.
- Format tiap pasal seperti contoh di bawah.
- Urutkan dari pasal paling utama.
- Jika tidak ditemukan, jawab: "Pasal tidak ditemukan dalam dokumen."

Contoh format jawaban:
Pasal 433: Pencemaran lisan di muka umum → Ancaman: penjara 9 bulan
Pasal 441: Pencemaran melalui tulisan/media → Ancaman: penjara 1 tahun 6 bulan

Jawaban:
""")

In [33]:
from langchain.chains import RetrievalQA
from langchain_ollama import ChatOllama

llm = ChatOllama(model="gemma3:4b", temperature=0.1)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=hybrid_retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt_template},
    return_source_documents=True
)
response = qa_chain.invoke(
    {"query": "Berapa denda maksimal kalau ketahuan main judi online??"}
)

print(response["result"])


Pasal 427: Setiap Orang yang menggunakan kesempatan main judi yang diadakan tanpa izin → Ancaman: pidana penjara paling lama 3 (tiga) tahun atau pidana denda paling banyak kategori III.

Pasal 426 (1): Setiap Orang yang tanpa izin menawarkan atau memberi kesempatan untuk main judi dan menjadikan sebagai mata pencaharian atau turut serta dalam perusahaan perjudian; menawarkan atau memberi kesempatan kepada umum untuk main judi atau turut serta dalam perusahaan perjudian, terlepas dari ada tidaknya suatu syarat atau tata cara yang harus dipenuhi untuk menggunakan kesempatan tersebut; menjadikan turut serta pada permainan judi sebagai mata pencaharian → Ancaman: pidana penjara paling lama 9 (sembilan) tahun atau pidana denda paling banyak kategori VI.

Pasal 84: Setiap Orang yang telah berulang kali dijatuhi pidana denda untuk Tindak Pidana yang hanya diancam dengan pidana denda paling banyak kategori II → Ancaman: pidana pengawasan paling lama 6 (enam) Bulan dan pidana denda yang diperbe

In [35]:
from langchain.chains import RetrievalQA
from langchain_ollama import ChatOllama

llm = ChatOllama(model="gemma3:4b", temperature=0.1)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=hybrid_retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt_template},
    return_source_documents=True
)
response = qa_chain.invoke(
    {"query": "Berapa hukuman maksimal kalau memukul orang dengan terencana dan orang tersebut meninggal??"}
)

print(response["result"])


Pasal 458 (1): Setiap Orang yang merampas nyawa orang lain, dipidana karena pembunuhan, dengan pidana penjara paling lama 15 (lima belas) tahun.
Pasal 458 (2): Jika Tindak Pidana sebagaimana dimaksud pada ayat (1) dilakukan terhadap ibu, Ayah, istri, suami, atau anaknya, pidananya dapat ditambah 1/3 (satu per tiga).
Pasal 462: Setiap Orang yang mendorong, membantu, atau memberi sarana kepada orang lain untuk bunuh diri dan orang tersebut mati karena bunuh diri, dipidana dengan pidana penjara paling lama 4 (empat) tahun.
Pasal 468 (2): Jika perbuatan sebagaimana dimaksud pada ayat (1) mati, dipidana dengan pidana penjara paling lama 10 (sepuluh) tahun.



In [36]:
from langchain.chains import RetrievalQA
from langchain_ollama import ChatOllama

llm = ChatOllama(model="gemma3:4b", temperature=0.1)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=hybrid_retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt_template},
    return_source_documents=True
)
response = qa_chain.invoke(
    {"query": "Joko mencuri hasil pertanian dengan terencana kira kira joko akan mendapatkan hukuman apa ???"}
    
)

print(response["result"])


Pasal 592: Setiap Orang yang menjadikan kebiasaan untuk membeli, menukar, menerima jaminan atau gadai, menyimpan, atau menyembunyikan benda yang diperoleh dari Tindak Pidana, dipidana dengan pidana penjara paling lama 6 (enam) tahun atau pidana denda paling banyak kategori V.

Pasal 15: Persiapan melakukan Tindak Pidana terjadi jika pelaku berusaha untuk mendapatkan atau menyiapkan sarana berupa alat, mengumpulkan informasi atau menyusun perencanaan tindakan, atau melakukan tindakan serupa yang dimaksudkan untuk menciptakan kondisi untuk dilakukannya suatu perbuatan yang secara langsung ditujukan bagi penyelesaian Tindak Pidana. Persiapan melakukan Tindak Pidana dipidana, jika ditentukan secara tegas dalam Undang-Undang. Pidana untuk persiapan melakukan Tindak Pidana paling banyak l/2 (satu per dua) dari maksimum ancaman pidana pokok untuk Tindak Pidana yang bersangkutan.



In [37]:
from langchain.chains import RetrievalQA
from langchain_ollama import ChatOllama

llm = ChatOllama(model="gemma3:4b", temperature=0.1)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=hybrid_retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt_template},
    return_source_documents=True
)
response = qa_chain.invoke(
    {"query": "apa itu restorasi justice ???"}
    
)

print(response["result"])


Pasal tidak ditemukan dalam dokumen.



In [42]:
from langchain.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template("""
Kamu adalah asisten hukum Indonesia yang membantu menjelaskan aturan KUHP dengan bahasa yang mudah dipahami.

Konteks (kutipan pasal KUHP):
{context}

Pertanyaan:
{question}

Instruksi:
- Jawab pertanyaan berdasarkan konteks di atas.
- Gunakan bahasa yang jelas dan mudah dipahami.
- Sertakan nomor pasal sebagai referensi di akhir jawaban.
- Jika konteks tidak relevan, jawab: "Informasi tidak ditemukan dalam dokumen."

Contoh:
Pertanyaan: Apakah menghina orang di depan umum bisa dipidana?
Jawaban: Ya, menghina orang secara lisan di depan umum dapat dikenakan pidana penjara hingga 9 bulan. Jika penghinaan dilakukan melalui tulisan atau media, ancamannya lebih berat yaitu hingga 1 tahun 6 bulan. [Ref: Pasal 433, Pasal 441]

Jawaban:
""")

In [43]:
from langchain.chains import RetrievalQA
from langchain_ollama import ChatOllama

llm = ChatOllama(model="gemma3:4b", temperature=0.1)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=hybrid_retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt_template},
    return_source_documents=True
)
response = qa_chain.invoke(
    {"query": "Apa hukuman untuk orang yang melakukan persetubuhan dengan orang yang bukan suami/istrinya???"}
    
)

print(response["result"])


Hukuman untuk orang yang melakukan persetubuhan dengan orang yang bukan suami/istrinya adalah sebagai berikut:

Pasal 411 (1) mengatur mengenai perzinaan. Jika seseorang melakukan persetubuhan dengan orang yang bukan suaminya atau istrinya, maka akan dipidana karena perzinaan. Pidana yang diterapkan adalah pidana penjara paling lama 1 (satu) tahun atau pidana denda paling banyak kategori II. [Ref: Pasal 411 ayat (1)]

Selain itu, terdapat juga Pasal 413 yang mengatur mengenai persetubuhan dengan anggota keluarga dekat (anak, saudara kandung, atau sepupu tingkat pertama). Dalam pasal ini, pelaku dapat dipidana dengan pidana penjara paling lama 10 (sepuluh) tahun. [Ref: Pasal 413]



In [47]:
from langchain.chains import RetrievalQA
from langchain_ollama import ChatOllama

llm = ChatOllama(model="gemma3:4b", temperature=0.1)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=hybrid_retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt_template},
    return_source_documents=True
)
response = qa_chain.invoke(
    {"query": "apa jenis hukuman yang bisa diberikan untuk anak ?"}
    
)

print(response["result"])


Berdasarkan konteks yang diberikan, jenis hukuman yang bisa diberikan untuk anak bervariasi tergantung pasal yang berlaku. Berikut rinciannya:

*   **Pasal 425 (1) dan (2):** Hukuman penjara paling lama 4 tahun atau denda paling banyak kategori IV jika memberikan anak di bawah 12 tahun kepada orang lain untuk dimanfaatkan dalam melakukan perbuatan meminta-minta atau pekerjaan berbahaya.
*   **Pasal 452 (1):** Hukuman penjara paling lama 6 tahun atau denda paling banyak kategori IV jika menarik anak dari kekuasaan sesuai peraturan. Jika dengan tipu muslihat, kekerasan, atau ancaman kekerasan, hukuman penjara paling lama 8 tahun atau denda paling banyak kategori V.
*   **Pasal 453 (1) dan (2):** Hukuman penjara paling lama 4 tahun atau denda paling banyak kategori III jika menyembunyikan atau menarik anak dari kekuasaan. Jika terhadap anak di bawah 12 tahun, hukuman penjara paling lama 7 tahun.
*   **Pasal 430:** Hukuman 1/2 dari pidana sebagaimana dimaksud dalam Pasal 429 ayat (1) dan (

In [44]:
from langchain.chains import RetrievalQA
from langchain_ollama import ChatOllama

llm = ChatOllama(model="gemma3:4b", temperature=0.1)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=hybrid_retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt_template},
    return_source_documents=True
)
response = qa_chain.invoke(
    {"query": "Berapa denda kalau mengganggu ibadah orang lain di tempat ibadah???"}
    
)

print(response["result"])


Mengganggu ibadah orang lain di tempat ibadah dapat dipidana dengan denda. Berdasarkan Pasal 303 (1) KUHP, jika seseorang membuat gaduh di dekat tempat untuk menjalankan ibadah pada waktu ibadah sedang berlangsung, maka akan dipidana dengan pidana denda paling banyak kategori I. 

[Ref: Pasal 303 (1) KUHP]



In [40]:
from langchain.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template("""
Kamu adalah asisten hukum Indonesia. Jawab pertanyaan berdasarkan konteks berikut dengan bahasa yang mudah dipahami. Jika tidak ada informasi yang relevan, jawab: "Informasi tidak tersedia."

Konteks:
{context}

Contoh:
Pertanyaan: Apakah menghina orang di depan umum bisa dipidana?
Jawaban: Ya, bisa dipidana penjara hingga 9 bulan. Jika lewat tulisan atau media, ancamannya hingga 1 tahun 6 bulan. [Pasal 433, 441]

Pertanyaan: {question}
Jawaban:
""")

In [41]:
from langchain.chains import RetrievalQA
from langchain_ollama import ChatOllama

llm = ChatOllama(model="gemma3:4b", temperature=0.1)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=hybrid_retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt_template},
    return_source_documents=True
)
response = qa_chain.invoke(
    {"query": "Apa hukuman untuk orang yang melakukan persetubuhan dengan orang yang bukan suami/istrinya???"}
    
)

print(response["result"])


Hukuman untuk melakukan persetubuhan dengan orang yang bukan suami/istrinya adalah:

*   **Persetubuhan di luar perkawinan:** Pidana penjara paling lama 1 (satu) tahun atau pidana denda paling banyak kategori II.
*   **Pengaduan:** Penuntutan hanya dilakukan atas pengaduan suami/istri atau orang tua (jika tidak ada perkawinan).

Informasi tidak tersedia.


## Testing

In [14]:
from langchain.chains import RetrievalQA
from langchain_ollama import ChatOllama

llm = ChatOllama(model="gemma3:4b", temperature=0.1)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=hybrid_retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt_template},
    return_source_documents=True
)
response = qa_chain.invoke(
    {"query": "Berapa denda maksimal kalau ketahuan main judi online??"}
)

print(response["result"])


Berdasarkan dokumen KUHP yang tersedia, berikut adalah analisis mengenai denda maksimal jika ketahuan main judi online:

**1. Pasal-pasal KUHP yang Relevan:**

*   **Pasal 427:** Setiap Orang yang menggunakan kesempatan main judi yang diadakan tanpa izin, dipidana dengan pidana penjara paling lama 3 (tiga) tahun atau pidana denda paling banyak kategori III.
*   **Pasal 78 (1):** Pidana denda merupakan sejumlah uang yang wajib dibayar oleh terpidana berdasarkan putusan pengadilan.
*   **Pasal 78 (2):** Jika tidak ditentukan minimum khusus, pidana denda ditetapkan paling sedikit Rp50.000,00 (lima puluh ribu rupiah).

**2. Unsur-unsur Pidana yang Terpenuhi:**

Kasus main judi online, berdasarkan Pasal 427, memerlukan unsur-unsur sebagai berikut:

*   **Keberadaan Tindak Permainan Judi:**  Main judi online secara umum memenuhi unsur ini.
*   **Penggunaan Kesempatan:** Terpidana menggunakan kesempatan yang disediakan dalam permainan judi online tersebut.
*   **Tanpa Izin:** Permainan judi o

In [16]:
import json
from tqdm import tqdm

# Load test data
with open("/Users/muhammadzuamaalamin/Documents/RisetTextMining/retrieval2/datates2.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

# Ekstrak semua field yang relevan dari data test
questions        = [item["question"]        for item in test_data]
references       = [item["answer"]          for item in test_data]
relevant_pasals  = [item["relevant_pasal"]  for item in test_data]
topics           = [item["topic"]           for item in test_data]


def run_qa(chain, question):
    response = chain.invoke({"query": question})
    return response["result"]


# Jalankan evaluasi per item
results = []

for item in tqdm(test_data, desc="Evaluasi QA", unit="soal", colour="green"):
    question       = item["question"]
    reference      = item["answer"]
    relevant_pasal = item["relevant_pasal"]
    topic          = item["topic"]

    # Jalankan chain
    predicted = run_qa(qa_chain, question)

    # Cek apakah pasal yang relevan muncul di jawaban model
    pasal_found  = [p for p in relevant_pasal if p in predicted]
    pasal_missed = [p for p in relevant_pasal if p not in predicted]

    pasal_acc = len(pasal_found) / len(relevant_pasal) if relevant_pasal else 0

    results.append({
        "topic"          : topic,
        "question"       : question,
        "reference"      : reference,
        "predicted"      : predicted,
        "relevant_pasal" : relevant_pasal,
        "pasal_found"    : pasal_found,
        "pasal_missed"   : pasal_missed,
        "pasal_accuracy" : pasal_acc
    })

    # Update tqdm postfix dengan info real-time
    tqdm.write(f"\n{'='*60}")
    tqdm.write(f"Topic   : {topic}")
    tqdm.write(f"Q       : {question}")
    tqdm.write(f"Pasal   : {relevant_pasal}")
    tqdm.write(f"Found   : {pasal_found}")
    tqdm.write(f"Missed  : {pasal_missed}")
    tqdm.write(f"Accuracy: {pasal_acc:.0%}")
    tqdm.write(f"Answer  : {predicted[:200]}...")


# Ringkasan akhir
total         = len(results)
avg_pasal_acc = sum(r["pasal_accuracy"] for r in results) / total

print(f"\n{'='*60}")
print(f"SUMMARY")
print(f"{'='*60}")
print(f"Total Questions  : {total}")
print(f"Avg Pasal Acc    : {avg_pasal_acc:.2%}")


# Simpan hasil ke file
with open("eval_resultsallminilm.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("\nHasil evaluasi disimpan ke eval_results.json")

Evaluasi QA:   4%|▍         | 1/25 [00:21<08:35, 21.49s/soal]


Topic   : Penghinaan dan Pencemaran Nama Baik
Q       : Kalau saya menghina orang di media sosial, ancamannya berapa tahun penjara?
Pasal   : ['433', '441']
Found   : ['433']
Missed  : ['441']
Accuracy: 50%
Answer  : Baik, berdasarkan pertanyaan Anda mengenai ancaman penjara jika menghina orang di media sosial, berikut analisisnya berdasarkan dokumen KUHP yang tersedia:

**1. Pasal-pasal KUHP yang Relevan beserta ...


Evaluasi QA:   8%|▊         | 2/25 [00:39<07:29, 19.56s/soal]


Topic   : Perjudian
Q       : Berapa denda maksimal kalau ketahuan main judi online?
Pasal   : ['426', '427']
Found   : ['427']
Missed  : ['426']
Accuracy: 50%
Answer  : Baik, berdasarkan dokumen KUHP yang Anda berikan, berikut adalah analisis mengenai denda maksimal jika ketahuan main judi online:

**1. Pasal-pasal KUHP yang Relevan:**

*   **Pasal 427** “Setiap Oran...


Evaluasi QA:  12%|█▏        | 3/25 [00:59<07:15, 19.80s/soal]

...


Topic   : Berita Bohong
Q       : Berapa denda kalau ketahuan menyebar berita bohong yang bikin rusuh?
Pasal   : ['263']
Found   : ['263']
Missed  : []
Accuracy: 100%
Answer  : Baik, berdasarkan dokumen KUHP yang Anda berikan, berikut analisis mengenai denda jika ketahuan menyebarkan berita bohong yang menyebabkan rusuh:

**1. Pasal-Pasal KUHP yang Relevan:**

*   **Pasal 26...


Evaluasi QA:  76%|███████▌  | 19/25 [07:14<02:01, 20.26s/soal]


Topic   : Memaksa Masuk Rumah
Q       : Apa hukuman untuk orang yang memaksa masuk ke rumah orang lain?
Pasal   : ['257']
Found   : ['257']
Missed  : []
Accuracy: 100%
Answer  : Baik, berdasarkan dokumen KUHP yang diberikan, berikut adalah analisis hukum untuk pertanyaan mengenai hukuman atas orang yang memaksa masuk ke rumah orang lain:

**1. Pasal-Pasal KUHP yang Relevan:**...


Evaluasi QA:  76%|███████▌  | 19/25 [07:34<02:23, 23.92s/soal]


KeyboardInterrupt: 

In [38]:
questions

['Apa yang dimaksud dengan tanaman tebu?',
 'Apa kepanjangan dari MBS dalam konteks tebu layak giling?',
 'Berapa kisaran curah hujan yang sesuai untuk budidaya tebu?',
 'Apa saja dua pola tanam tebu yang disebutkan dalam pedoman?',
 'Apa fungsi utama kegiatan klentek pada tanaman tebu?',
 'Berapa minimal kebutuhan benih bagal mata 2-3 per hektar?',
 'Apa yang dimaksud dengan Plant Cane (PC)?',
 'Berapa minimal umur tebu agar layak giling?',
 'Alat apa yang digunakan untuk mengukur kedalaman permukaan air tanah?',
 'Penyakit apa yang disebabkan oleh bakteri Leaf sonia xyli subsp.xyli?',
 'Apa yang dimaksud dengan varietas tebu unggul?',
 'Sebutkan dua bentuk benih tebu yang dapat digunakan.',
 'Apa nama sistem pengolahan tanah pada lahan sawah yang dikerjakan secara manual dengan pembuatan saluran air?',
 'Apa fungsi pekerjaan gulud pada budidaya tebu?',
 'Hama apakah yang gejala serangannya ditandai dengan cambuk hitam pada pucuk tanaman?',
 'Berapa batas toleransi kadar kotoran pada 

In [39]:
references

['Tanaman tebu adalah jenis tanaman semusim yang mengandung sukrosa atau yang mengandung kadar gula dan dibudidayakan untuk bahan baku pabrik gula.',
 'Masak, Bersih, dan Segar.',
 'Curah hujan antara 1.000–2.000 milimeter per tahun dengan sekurang-kurangnya 3 bulan kering.',
 'Pola A/I (lahan berpengairan, tanam April-Agustus) dan Pola B/II (lahan tadah hujan, tanam September-November).',
 'Untuk mengelupas daun tebu yang telah kering atau kuning, melancarkan sirkulasi udara dan cahaya, serta mengurangi kelembaban.',
 'Minimal 60.000 mata per hektar.',
 'Tanaman tebu pertama yang berasal dari lahan bukan bekas tebu, menggunakan benih unggul dan bersertifikat.',
 'Minimal 11 (sebelas) bulan.',
 'Pipa Piezometer.',
 'Penyakit Pembuluh Ratoon Stunting Disease (RSD).',
 'Varietas tebu yang menunjukkan adaptasi dan produktivitas yang tinggi, serta memiliki keunggulan-keunggulan tertentu baik dari aspek keragaan tanaman maupun parameter pabrikasi.',
 'Benih berupa setek batang/bagal mata 2 

In [22]:
import numpy as np
from tqdm import tqdm
from rouge_score import rouge_scorer
from bert_score import score as bert_score


# ── 1. Pastikan predictions & references diambil dari results ──────────────
predictions = [r["predicted"]  for r in results]
references  = [r["reference"]  for r in results]


# ── 2. ROUGE-L ─────────────────────────────────────────────────────────────
def evaluate_rouge_l(predictions, references):
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    scores = []

    for pred, ref in tqdm(zip(predictions, references),
                          total=len(predictions),
                          desc="ROUGE-L",
                          colour="cyan"):
        score = scorer.score(ref, pred)
        scores.append(score["rougeL"].fmeasure)

    return {
        "rougeL_mean"  : np.mean(scores),
        "rougeL_scores": scores
    }


# ── 3. BERTScore ───────────────────────────────────────────────────────────
def evaluate_bertscore(predictions, references):
    P, R, F1 = bert_score(
        predictions,
        references,
        lang="id",
        model_type="bert-base-multilingual-cased",
        verbose=True
    )

    return {
        "precision": P.mean().item(),
        "recall"   : R.mean().item(),
        "f1"       : F1.mean().item(),
        "f1_scores": F1.tolist()          # per-item score
    }


# ── 4. Jalankan evaluasi ───────────────────────────────────────────────────
print("\n" + "="*60)
print("MENGHITUNG ROUGE-L ...")
print("="*60)
rouge_result = evaluate_rouge_l(predictions, references)

print("\n" + "="*60)
print("MENGHITUNG BERTScore ...")
print("="*60)
bertscore_result = evaluate_bertscore(predictions, references)



MENGHITUNG ROUGE-L ...


ROUGE-L: 100%|██████████| 25/25 [00:00<00:00, 190.80it/s]


MENGHITUNG BERTScore ...


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 4.95 seconds, 5.05 sentences/sec


In [ ]:
# ── 5. Tampilkan ringkasan ─────────────────────────────────────────────────
print("\n" + "="*60)
print("HASIL EVALUASI LENGKAP")
print("="*60)
print(f"Pasal Accuracy (avg) : {avg_pasal_acc:.2%}")
print(f"ROUGE-L (mean)       : {rouge_result['rougeL_mean']:.4f}")
print(f"BERTScore Precision  : {bertscore_result['precision']:.4f}")
print(f"BERTScore Recall     : {bertscore_result['recall']:.4f}")
print(f"BERTScore F1         : {bertscore_result['f1']:.4f}")


# ── 6. Gabungkan semua metric ke results & simpan ─────────────────────────
for i, r in enumerate(results):
    r["rougeL"]          = rouge_result["rougeL_scores"][i]
    r["bertscore_f1"]    = bertscore_result["f1_scores"][i]

with open("eval_results.json", "w", encoding="utf-8") as f:
    import json
    json.dump({
        "summary": {
            "total_questions"   : len(results),
            "avg_pasal_accuracy": avg_pasal_acc,
            "rougeL_mean"       : rouge_result["rougeL_mean"],
            "bertscore_precision": bertscore_result["precision"],
            "bertscore_recall"  : bertscore_result["recall"],
            "bertscore_f1"      : bertscore_result["f1"]
        },
        "details": results
    }, f, ensure_ascii=False, indent=2)

print("\nHasil lengkap disimpan ke eval_results.json")


HASIL EVALUASI LENGKAP
Pasal Accuracy (avg) : 70.00%
ROUGE-L (mean)       : 0.1648
BERTScore Precision  : 0.6169
BERTScore Recall     : 0.7632
BERTScore F1         : 0.6820

Hasil lengkap disimpan ke eval_results.json
